# LangChain工具调用实践指南

## 一、工具调用基础概念

在LangChain框架中，工具(Tool)是扩展大语言模型能力的关键组件。工具调用允许LLM：
- 访问实时信息（如网络搜索）
- 执行精确计算
- 获取系统状态
- 与外部API交互

## 二、工具调用示例

### 1 创建获取当前时间工具

#### 1.1 创建基础工具

首先我们创建一个获取当前时间的简单工具：

In [5]:
import os
from datetime import datetime
from langchain.agents import tool

@tool
def get_current_time():
    """返回当前日期和时间，格式为YYYY-MM-DD HH:MM:SS"""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

#### 1.2 初始化模型和工具

In [6]:
from langchain_deepseek import ChatDeepSeek
from dotenv import load_dotenv, find_dotenv

def load_deepseek_config():
    """安全加载DeepSeek API配置
    自动查找并加载项目中的.env文件，保护敏感配置信息
    """
    load_dotenv(find_dotenv())

load_deepseek_config() # 初始化环境变量

# 初始化DeepSeek模型
llm = ChatDeepSeek(
    model="deepseek-chat",
    temperature=0.0,
    max_retries=2
)

# 工具列表
tools = [get_current_time]

#### 1.3 创建代理并测试

In [7]:
from langgraph.prebuilt import create_react_agent
from langchain_core.runnables import RunnableLambda
from langchain_core.messages import HumanMessage

# 创建代理
agent = create_react_agent(
    tools=tools,
    model=llm
)
# 创建输入转换器
input_adapter = RunnableLambda(
    lambda x: {"messages": [HumanMessage(content=x["input"])]}
)
chain = input_adapter | agent

# 测试调用
response = chain.invoke({"input": "现在是什么时间？"})
print(response["messages"][-1].content)

现在是2025年4月9日，中午12点41分。


### 3 创建数学计算器工具

#### 3.1 创建计算器

In [8]:
from langchain_community.tools import Tool

calculator = Tool(
    name="calculator",
    func=lambda x: str(eval(x)),  # 注意安全风险，实际项目应做输入验证
    description="执行数学计算，支持加减乘除和括号"
)

#### 3.2 测试计算器功能

In [9]:
agent = create_react_agent(
    tools=[calculator],
    model=llm
)

chain = input_adapter | agent

response = chain.invoke({"input": "计算(125 + 37)*2.5等于多少？"})
print(response["messages"][-1].content)

计算结果为405.0。


### 3 创建网络搜索工具

#### 3.1 集成DuckDuckGO搜索

In [10]:
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun(
    name="web_search",
    description="查询最新信息，包括新闻、百科等"
)

#### 3.2 测试网络搜索工具功能

In [11]:
agent = create_react_agent(
    tools=[search],
    model=llm
)

chain = input_adapter | agent

response = chain.invoke({"input": "请简要概述一下哪吒2的内容"})
print(response["messages"][-1].content)

《哪吒2》（全称《哪吒2·魔童闹海》）的故事延续了前作的剧情，主要讲述了哪吒和敖丙在天劫之后灵魂得以保留，但肉身即将消散。为了拯救他们，太乙真人决定使用七色宝莲为两人重塑肉身。然而，这一过程充满了挑战，包括申公豹的阴谋和四龙王的威胁。

影片还揭示了封神大战背后的阴谋，以及神仙之间的亲情与矛盾。申公豹的角色反转成为一大亮点，他出身妖族，因遭受歧视而被迫为无量仙翁卖命，最终选择反抗。

《哪吒2》以其精彩的剧情、深刻的角色塑造和丰富的文化内涵，成为春节档的热门电影，票房和口碑双丰收。


### 4. 组合多个工具

在LangChain中，Agent运用ReAct（推理Reasoning + 行动Acting）框架，能根据任务需求自动选合适工具。模型负责通过推理来选择合适的工具，ReAct框架调用模型选择的工具并给出结果反馈，形成“思考-行动-验证-调整”闭环决策，帮助大语言模型（LLM）高效利用多种工具。 简化逻辑如下：[LangChainAgentReAct.py](source/re_act/LangChainAgentReAct.py)

In [12]:
tools = [get_current_time, calculator, search]

agent = create_react_agent(
    tools=tools,
    model=llm
)

chain = input_adapter | agent

response = chain.invoke({
    "input": "今天日期是多少？再搜索下今天的重要新闻"
})

for msg in response["messages"]:
    print(f"[{msg.type}]: {msg.content}")

[human]: 今天日期是多少？再搜索下今天的重要新闻
[ai]: 
[tool]: 2025-04-09 12:41:46
[tool]: 今日摘要习近平总书记参加首都义务植树活动时的重要讲话引发热烈反响。广大干部群众表示，要牢记总书记嘱托，践行绿色发展理念，弘扬生态 ... 今日摘要习近平致电祝贺米拉诺维奇当选连任克罗地亚总统。经初步核算，2024年我国国内生产总值达134.9万亿元，比上年增长5.0%，主要目标任务顺利 ... 转自：财联社【4月4日周五《新闻联播》要闻15条】财联社4月4日电，今天《新闻联播》主要内容有： 1、汇聚共建美丽中国磅礴力量——习近平总 ... 每天一起跟大家看新鲜新闻，恳请您动动手指，为小编送上一份免费的关注与爱心。您的支持是小编不懈更新的动力。今天是2024年11月25日!今日要闻有：1、我国在酒泉卫星发射中心用长征二号丙运载火箭成功发射四维高景二号 03、04 星。 今日国内 10 大新闻总结 ... 在地面站接收首幅影像，其进入在轨测试阶段，预计明年汛前投入使用，将为水利工作提供重要 ... 逼迫喝尿，孕期被强行发生性关系，曾在4个月内两度流产 4月2日，在郑州一居民小区，大皖新闻记者见到了今年36岁的李萍。李萍介绍 ...
[ai]: 今天的日期是 **2025年4月9日**。

### 今日重要新闻摘要：
1. **习近平总书记参加首都义务植树活动**，强调绿色发展理念，引发热烈反响。
2. **习近平致电祝贺米拉诺维奇当选连任克罗地亚总统**。
3. **2024年我国国内生产总值达134.9万亿元**，同比增长5.0%，主要目标任务顺利完成。
4. **我国成功发射四维高景二号03、04星**，将为水利工作提供重要支持。
5. 其他社会新闻，如郑州居民李萍的遭遇引发关注。

如需了解更多详情，可以进一步查询具体新闻来源。


## 三、工具调用监控与分析

### 1 提取工具调用信息

In [13]:
def analyze_tool_usage(response):
    tool_log = []

    for msg in response["messages"]:
        if hasattr(msg, "tool_calls"):
            for call in msg.tool_calls:
                tool_log.append({
                    "action": "INVOKE",
                    "tool": call["name"],
                    "input": call["args"],
                    "id": call["id"]
                })
        elif msg.type == "tool":
            tool_log.append({
                "action": "RESULT",
                "tool": msg.name,
                "output": msg.content,
                "id": msg.tool_call_id
            })

    return tool_log

### 2 查看调用详情

In [14]:
response = chain.invoke({
    "input": "今天日期是多少？再搜索下今天的重要新闻"
})

usage = analyze_tool_usage(response)

for item in usage:
    print(f"{item['action']} {item['tool']}")
    if "input" in item:
        print(f"  Input: {item['input']}")
    if "output" in item:
        print(f"  Output: {item['output'][:100]}...")  # 截断长输出
    print()

INVOKE get_current_time
  Input: {}

INVOKE web_search
  Input: {'query': '今天的重要新闻'}

RESULT get_current_time
  Output: 2025-04-09 12:42:02...

RESULT web_search
  Output: 美国之音即時准确报道全球重大新闻，从政治到娱乐，从尖端科技到英语学习，内容包罗万象，应有尽有。欢迎浏览美国之音中文网VOAChinese.com 阅读 ... 世界新聞網提供全球華人關心的即時新聞；除...



## 四、综合应用示例
全量代码：[python-practice/tool_invoke.py](python-practice/tool_invoke.py)

```
输入：
今天南京天气如何？请帮我根据天气制定一个南京的游玩攻略，再帮我计算去那里旅游预算：5天住宿每天400元，机票往返2000元

输出：
### 南京游玩攻略（根据当前天气）

#### 1. **天气情况**
- **今日天气**：阴转多云，气温17℃-30℃，紫外线较强，空气质量良好。
- **建议**：白天注意防晒，早晚温差较大，建议携带薄外套。

#### 2. **游玩推荐**
- **上午**：
  - **玄武湖公园**：适合晨练或散步，清晨气温较低，建议穿薄外套。
  - **南京博物院**：室内活动，避开中午高温时段。
- **下午**：
  - **南京金牛湖风景区**：适合户外活动，但注意防晒。
  - **夫子庙**：傍晚时分气温舒适，适合逛街和品尝小吃。
- **晚上**：
  - **夜游秦淮河**：气温降至17℃，建议增添衣物，享受夜景。

#### 3. **预算计算**
- **住宿**：5天 × 400元/天 = **2000元**
- **机票**：往返 **2000元**
- **总计**：2000元（住宿） + 2000元（机票） = **4000元**

#### 4. **其他建议**
- **携带物品**：防晒霜、遮阳帽、薄外套、水杯、墨镜。
- **交通**：能见度良好，适合出行。
...
- INVOKE get_city_spot
- RESULT get_city_spot
- INVOKE get_city_weather
- RESULT get_city_weather
```

```mermaid
sequenceDiagram
    participant User
    participant Agent
    participant LLM
    participant Tools

    User->>Agent: "南京天气+攻略+预算"
    Agent->>LLM: 发送初始Prompt，包含工具描述和用户问题

    rect rgba(240, 255, 240, 0.5)
        LLM->>Agent: "Action: get_city_weather<br>Action Input: 南京"
        Agent->>Tools: 调用天气工具
        Tools->>Agent: 返回天气数据
        Agent->>LLM: "拼接Observation: 天气数据"
        note over Agent,LLM: 查询景点信息方式相同（略）
    end
    

    rect rgba(255, 240, 240, 0.5)
        LLM->>Agent: "Action: calculator<br>Action Input: 5*400+2000"
        Agent->>Tools: 调用计算器
        Tools->>Agent: 返回4000
        Agent->>LLM: "拼接Observation: 4000"
    end

    LLM->>Agent: 生成最终自然语言回复
    Agent->>User: 整合后的答案   
```
### 1 创建天气查询工具

#### 1.1 创建城市信息摘取工具

In [15]:
from langchain_core.prompts import ChatPromptTemplate

def get_completion_from_messages(messages, temperature=0):
    llm.temperature = temperature
    response = llm.invoke(messages)
    return response.content


def get_city(text):
    template_string = """
    请从用户输入中精确提取中国地级市及下属区县名称，并严格按以下规则输出：

    **提取规则：**
    1. 必须包含市级和区县级两个行政单位
    2. 输出格式为："xx市xx区" 或 "xx市xx县"
    3. 直辖市特殊处理（如北京、上海等），格式为："xx区"（不重复市级）

    **处理要求：**
    - 若原文同时包含区和县，优先保留区
    - 若原文只有市级名称，补充该市的主城区（如"南京市"→"南京市玄武区"）
    - 若原文不包含有效信息，返回"未识别"
    - 返回结果仅包含xx市xx区或xx市xx县，不要附加任何解释信息

    **示例：**
    输入："明天江宁的天气怎么样"
    输出："南京市江宁区"

    输入："我想查询上海浦东新区的交通"
    输出："浦东新区"

    输入："河北省邯郸市涉县天气预报"
    输出："邯郸市涉县"

    **请处理以下输入：**
    "{text}"
     """
    template = ChatPromptTemplate.from_template(template_string)
    messages = template.format_messages(text=text)
    return get_completion_from_messages(messages=messages)

print(get_city("我周末想要去江宁区游玩"))

"南京市江宁区"


#### 1.2 创建城市编码查询工具
(该工具方法涉及文件内容读取及向量存储与检索，具体内容将在下一章节详加阐述)

In [16]:
from langchain_community.document_loaders import CSVLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.indexes import VectorstoreIndexCreator
from langchain_community.vectorstores import DocArrayInMemorySearch

def get_city_code(text):
    """
    返回指定城市、区县的编码信息
    """
    city_name = get_city(text) # 用于从文本中摘取城市以及区县信息
    file = "./file/5_city_code.csv"
    loader = CSVLoader(file_path=file, encoding="gbk")
    # 定义 embedding
    embedding = HuggingFaceEmbeddings(model_name="BAAI/bge-small-zh-v1.5")  # 中文优化模型
    # 基于文档加载器创建向量存储
    index = VectorstoreIndexCreator(embedding=embedding, vectorstore_cls=DocArrayInMemorySearch).from_loaders([loader])

    query = f"""
    请检索出目标城市或区县的信息，并直接输出城市编号pid\

    城市名称{city_name}\

    当输入中既包含城市又包含区县时，只需要返回区县的信息即可
    """
    # 使用索引查询创建一个响应，并传入这个查询
    return index.query(query, llm=llm)

print(get_city_code("我周末想要去江宁区游玩"))

D:\ProgramData\anaconda3\envs\langchain\lib\site-packages\pydantic\_migration.py:283: UserWarning: `pydantic.error_wrappers:ValidationError` has been moved to `pydantic:ValidationError`.
  warnings.warn(f'`{import_path}` has been moved to `{new_location}`.')


320115000000


#### 1.3 根据城市编码调取API查询天气并总结天气信息返回结果
本案例使用的天气API参考于 http://tianqiapi.com/, 新注册的账号有免费的API调用次数，用于学习测试已经足够了。

In [17]:
import requests
import os


def weather_description(data):
    template_string = """
    你是一位专业的天气播报员，请根据以下JSON格式的天气数据，为游客生成简洁明了的天气简报和出行建议。要求：

    1. **核心天气概况**（50字以内）：
       - 用一句话总结今日整体天气状况
       - 标注当前实时温度和体感关键词（如"凉爽"、"闷热"等）

    2. **重点指标**（表格呈现）：
       | 指标        | 日间数值 | 夜间数值 |
       |-------------|----------|----------|
       | 温度范围    | [白天高温]℃ | [夜间低温]℃ |
       | 天气现象    | [白天天气] | [夜间天气] |
       | 降水概率    | [降雨概率]% | [降雨概率]% |
       | 紫外线强度  | [UV指数] - [强度描述] | - |
       | 空气质量    | [AQI] - [污染等级] | - |

    3. **分时段建议**：
       - 清晨（[日出时间]）：[活动建议]
       - 日间（10:00-16:00）：[活动建议]
       - 傍晚（[日落时间]前后）：[活动建议]
       - 夜间：[活动建议]

    4. **特殊提示**（如有）：
       - 空气质量提醒：[空气提示]
       - 极端天气预警：[警报内容]
       - 穿衣指南：根据温度变化推荐

    5. **旅游规划建议**：
       - 户外活动适宜度：⭐️[1-5星评价]
       - 推荐携带物品：[列出3-5项]
       - 交通影响：[风速和能见度对出行的影响]

    请基于以下数据生成报告（保持专业但友好的语气）：
    {data}
     """
    template = ChatPromptTemplate.from_template(template_string)
    messages = template.format_messages(data=data)
    return get_completion_from_messages(messages=messages)

@tool
def get_city_weather(text):
    """
    根据描述信息编码返回信息中指定城市、区县的天气信息
    """
    pid = get_city_code(text)
    weather_id = os.environ["weather_id"]
    weather_secret = os.environ["weather_secret"]
    # 调用第三方天气API（示例使用假想API）
    api_url = f"http://gfeljm.tianqiapi.com/api?unescape=1&version=v63&appid={weather_id}&appsecret={weather_secret}&adcode={pid}"
    response = requests.get(api_url)
    if response.status_code == 200:
        data = response.json()
        return weather_description(data)
    return "无法获取天气信息"

print(get_city_weather("我周末想要去江宁区游玩"))

C:\Users\Administrator\AppData\Local\Temp\ipykernel_252\1420543141.py:61: LangChainDeprecationWarning: The method `BaseTool.__call__` was deprecated in langchain-core 0.1.47 and will be removed in 1.0. Use :meth:`~invoke` instead.
  print(get_city_weather("我周末想要去江宁区游玩"))


**江宁天气简报**  
2025年4月9日 星期三 | 更新于12:05  

---

### 1. 核心天气概况  
今日阴天为主，当前温度26.4℃，体感**温暖舒适**，夜间转多云，温差较大（17-30℃）。  

---

### 2. 重点指标  
| 指标        | 日间数值       | 夜间数值       |
|-------------|----------------|----------------|
| 温度范围    | 30℃            | 17℃            |
| 天气现象    | 阴             | 多云           |
| 降水概率    | 0%             | 0%             |
| 紫外线强度  | 6 - 强         | -              |
| 空气质量    | 107 - 轻度污染 | -              |

---

### 3. 分时段建议  
- **清晨（05:42日出）**：适宜晨练，建议薄外套防凉。  
- **日间（10:00-16:00）**：紫外线较强，需防晒；户外活动避开正午高温。  
- **傍晚（18:29日落前后）**：散步好时机，注意夜间降温。  
- **夜间**：多云凉爽，建议室内活动或轻便长袖。  

---

### 4. 特殊提示  
- **空气质量**：轻度污染（PM10中度），敏感人群减少长时间户外停留。  
- **穿衣指南**：洋葱式穿搭（短袖+薄外套），昼夜温差达13℃。  

---

### 5. 旅游规划建议  
- **户外适宜度**：⭐️⭐️⭐️⭐️（4星，注意防晒）  
- **推荐携带**：防晒霜、遮阳帽、水壶、薄外套、晴雨伞（夜间防小雨）。  
- **交通影响**：能见度良好（15km），西风3级（12km/h），无显著影响。  

祝您在江宁旅途愉快！ 🌤️


### 2 创建景点查询工具

In [19]:
@tool
def get_city_spot(text):
    """
    返回指定城市、区县的编码信息
    """
    city_name = get_city(text) # 用于从文本中摘取城市以及区县信息
    file = "./file/5_city_spot.csv"
    loader = CSVLoader(file_path=file, encoding="gbk")
    # 定义 embedding
    embedding = HuggingFaceEmbeddings(model_name="BAAI/bge-small-zh-v1.5")  # 中文优化模型
    # 基于文档加载器创建向量存储
    index = VectorstoreIndexCreator(embedding=embedding, vectorstore_cls=DocArrayInMemorySearch).from_loaders([loader])

    query = f"""
    请执行以下操作：
    1. 检索{city_name}的所有景点信息
    2. 将所有景点名称提取出来
    3. 用中文顿号"、"连接成一个字符串
    4. 直接返回连接后的字符串，不要包含其他说明文字

    注意：
    - 如果{city_name}同时匹配到城市和区县，优先返回区县的景点
    - 确保每个景点名称之间用顿号分隔
    - 不要包含编号或其他字段
    """
    # 使用索引查询创建一个响应，并传入这个查询
    return index.query(query, llm=llm)

print(get_city_spot("我周末想要去江宁区游玩"))

南京天生桥风景名胜区、南京金牛湖风景区、大金山风景区


### 3 整合完整工作流

In [20]:
tools = [get_current_time, calculator, search, get_city_weather, get_city_spot]

agent = create_react_agent(
    tools=tools,
    model=llm
)

chain = input_adapter | agent

response = chain.invoke({
    "input": "今天南京天气如何？请帮我根据天气制定一个南京的游玩攻略，再帮我计算去那里旅游预算：5天住宿每天400元，机票往返2000元"
})

print("最终回答:", response["messages"][-1].content)
print("\n工具调用记录:")
for item in analyze_tool_usage(response):
    print(f"- {item['action']} {item['tool']}")

最终回答: ### 南京游玩攻略（根据当前天气）

#### 1. **天气情况**
- **今日天气**：阴转多云，气温17℃-30℃，紫外线较强，空气质量良好。
- **建议**：白天注意防晒，早晚温差较大，建议携带薄外套。

#### 2. **游玩推荐**
- **上午**：
  - **玄武湖公园**：适合晨练或散步，清晨气温较低，建议穿薄外套。
  - **南京博物院**：室内活动，避开中午高温时段。
- **下午**：
  - **南京金牛湖风景区**：适合户外活动，但注意防晒。
  - **夫子庙**：傍晚时分气温舒适，适合逛街和品尝小吃。
- **晚上**：
  - **夜游秦淮河**：气温降至17℃，建议增添衣物，享受夜景。

#### 3. **预算计算**
- **住宿**：5天 × 400元/天 = **2000元**
- **机票**：往返 **2000元**
- **总计**：2000元（住宿） + 2000元（机票） = **4000元**

#### 4. **其他建议**
- **携带物品**：防晒霜、遮阳帽、薄外套、水杯、墨镜。
- **交通**：能见度良好，适合出行。

祝您在南京玩得愉快！ 🌞

工具调用记录:
- INVOKE get_city_spot
- RESULT get_city_spot
- INVOKE get_city_weather
- RESULT get_city_weather
